# VulcanBench-SWE v4 comparison audit

## tl;dr

This notebook independently recomputes the ten model/effort aggregates and checks raw token receipts. The card includes Fable solver fallbacks and Claude reviewer fallbacks. No solver results are changed.

## Context & Methods

230 runs: two solver/harness combinations, five efforts, 23 matched tasks each. Code quality is the equal mean of three Astra and three Claude ratings. Combined = 50% functional + 15% automated quality + 15% security + 20% Code quality.

### Key Assumptions

CLI usage receipts are the token-accounting authority. Claude's per-turn usage is summed; its session cost is cumulative. Cached input is counted once as input, not as a second category on top of total input. Native CLI auxiliary accounting can differ. LLM reviewers may be biased; SE describes variation across tasks, not judge uncertainty.

This companion runs offline with the Python standard library. The generator executes all code cells top-to-bottom without a Jupyter kernel and retains their output.

## Data

### 1. Load audited evidence

Source: `comparison.json` next to this notebook. Raw paths and SHA-256 bindings are retained per run. Run from this output folder or from the repository root.

In [1]:
import hashlib
import json
import math
import statistics
from pathlib import Path

folder = Path.cwd()
if not (folder / "comparison.json").exists():
    folder = folder / "docs/results/swe-v4-astra-fable51-2026-09"
data = json.loads((folder / "comparison.json").read_text())
assert data["complete"] and data["runs"] == 230 and data["valid_votes"] == 1380
assert data["weights"] == {"functional": 0.5, "quality": 0.15, "security": 0.15, "human_like": 0.2}
rows = data["rows"]
assert len({(r["model"], r["effort"], r["task"]) for r in rows}) == 230
assert len({r["task"] for r in rows}) == 23
print(f"Verified population: {len(rows)} runs, {data['valid_votes']} ratings")
print(f"Fable solver fallback runs: {sum(r['fallback'] for r in rows)}")
print(f"Claude reviewer fallback ratings: {data['reviewer_fallback_votes']}")

Verified population: 230 runs, 1380 ratings
Fable solver fallback runs: 11
Claude reviewer fallback ratings: 10


## Results

### 2. Recompute scores and error bars

All rows use the same task set and fixed weights. Full passes require functional = 1.0; partial functional scores remain partial.

In [2]:
print("Model   Effort       Combined  Quality  Mean min  Passed")
for group in data["groups"]:
    subset = [r for r in rows if r["model"] == group["model"] and r["effort"] == group["effort"]]
    assert len(subset) == 23
    values = []
    for row in subset:
        quality = (row["astra"] + row["claude"]) / 2
        score = (
            0.5 * row["functional"] + 0.15 * row["quality"] + 0.15 * row["security"] + 0.2 * quality
        )
        assert math.isclose(score, row["combined"], abs_tol=1e-12)
        values.append(100 * score)
    assert math.isclose(statistics.mean(values), group["combined"]["mean"], abs_tol=1e-12)
    assert math.isclose(
        statistics.stdev(values) / math.sqrt(23), group["combined"]["se"], abs_tol=1e-12
    )
    print(
        f"{group['model']:7} {group['effort']:12} {statistics.mean(values):8.2f}  "
        f"{group['panel']['mean']:7.2f}  {group['minutes']['mean']:8.2f}  {group['passed']:2}/23"
    )

Model   Effort       Combined  Quality  Mean min  Passed
astra   low             91.60    81.89      4.22  22/23
astra   medium          91.73    83.04      3.82  23/23
astra   high            91.59    85.18      4.90  23/23
astra   extra-high      92.39    86.17      8.09  23/23
astra   max             92.25    86.54     10.28  23/23
fable   low             91.04    80.72     26.90  19/23
fable   medium          91.56    82.43     31.72  20/23
fable   high            90.97    83.81     28.76  20/23
fable   extra-high      91.53    84.96     38.77  22/23
fable   max             92.90    85.36     27.10  23/23


### 3. Check original token receipts

Fable's old summary field represents cache-price-weighted units from its last result. It is not comparable to Astra's raw token total. The card uses summed raw CLI usage instead. All original streams must still match the recorded hash.

In [3]:
totals = {"astra": 0, "fable": 0}
multi_result = 0
for row in rows:
    path = Path(row["source_directory"]) / "cli-agent-stream.jsonl"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["solver_receipt"]["stream_sha256"]
    events = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if row["model"] == "astra":
        usage = [e["usage"] for e in events if e.get("type") == "turn.completed"]
        assert len(usage) == 1
        total = usage[0]["input_tokens"] + usage[0]["output_tokens"]
    else:
        receipts = [e for e in events if e.get("type") == "result"]
        multi_result += len(receipts) > 1
        total = sum(
            e["usage"][key]
            for e in receipts
            for key in (
                "input_tokens",
                "cache_read_input_tokens",
                "cache_creation_input_tokens",
                "output_tokens",
            )
        )
    assert total == row["solver_receipt"]["raw_tokens"]
    totals[row["model"]] += total
for model, total in totals.items():
    seconds = sum(r["duration_s"] for r in rows if r["model"] == model)
    print(f"{model}: {total:,} raw tokens; {seconds / 3600:.4f} summed solver hours")
print(f"Fable runs with multiple usage receipts: {multi_result}")

astra: 97,043,705 raw tokens; 12.0030 summed solver hours
fable: 738,373,965 raw tokens; 58.7456 summed solver hours
Fable runs with multiple usage receipts: 4


## Takeaways

Use the paired component scores and runtime, not full-pass counts alone. Fable solver and Claude reviewer fallbacks remain included and separately disclosed. Historical summaries are preserved, but their unlike token units are not compared on the card. See `reviewer-accounting.json` for post-hoc judging time, usage, retries, and raw call provenance.